In [1]:
# Data Load
import pandas as pd
!pip install openpyxl

In [2]:

file_path = "./SAP-DataSet.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

['KNA1', 'LFA1', 'VBAK', 'VBAP', 'LIKP', 'LIPS', 'VTTK', 'VTTP']


In [3]:
Customers = pd.read_excel(excel_file,sheet_name="KNA1")
Carriers = pd.read_excel(excel_file,sheet_name="LFA1")
Orders = pd.read_excel(excel_file,sheet_name="VBAK")
OrderItems = pd.read_excel(excel_file,sheet_name="VBAP")
Deliveries = pd.read_excel(excel_file,sheet_name="LIKP")
DeliveryItems = pd.read_excel(excel_file,sheet_name="LIPS")
Shipments = pd.read_excel(excel_file,sheet_name="VTTK")
ShipmentItems = pd.read_excel(excel_file,sheet_name="VTTP")

In [4]:
# Standardize Columns
tables = [
    Customers,
    Carriers,
    Orders,
    OrderItems,
    Deliveries,
    DeliveryItems,
    Shipments,
    ShipmentItems
]

for df in tables:
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
    )

In [5]:
# Validation
for name,df in {
    "Customers":Customers,
    "Orders":Orders,
    "OrderItems":OrderItems,
    "Deliveries":Deliveries,
    "DeliveryItems":DeliveryItems,
    "Shipments":Shipments,
    "ShipmentItems":ShipmentItems
}.items():

    print("\n",name)
    print(df.isnull().sum())


 Customers
Customer_ID             0
Customer_Name           0
Country                 0
Region                  0
City                    0
Postal_Code             0
Street_Address          0
Phone_Number            0
Email_Address           0
Language                0
Tax_Number              0
Customer_Group          0
Sales_Organization      0
Distribution_Channel    0
Division                0
dtype: int64

 Orders
Sales_Document          0
Order_Date              0
Customer_ID             0
Order_Type              0
Sales_Organization      0
Distribution_Channel    0
Division                0
Order_Status            0
dtype: int64

 OrderItems
Sales_Document     0
Item_Number        0
Material_Number    0
Quantity           0
Net_Price          0
Item_Status        0
Delivery_Date      0
dtype: int64

 Deliveries
Delivery_Number      0
Delivery_Date        0
Sales_Document       0
Shipping_Point       0
Shipping_Type        0
Delivery_Status      0
Shipping_Status      0
Route   

In [6]:
# Duplicate Check
for name,df in {
    "Customers":Customers,
    "Orders":Orders,
    "OrderItems":OrderItems,
    "Deliveries":Deliveries,
    "DeliveryItems":DeliveryItems,
    "Shipments":Shipments,
    "ShipmentItems":ShipmentItems
}.items():

    print(name,df.duplicated().sum())

Customers 0
Orders 0
OrderItems 0
Deliveries 0
DeliveryItems 0
Shipments 0
ShipmentItems 0


In [7]:
# Date Conversion
Orders['Order_Date'] = pd.to_datetime(
    Orders['Order_Date']
)

OrderItems['Delivery_Date'] = pd.to_datetime(
    OrderItems['Delivery_Date']
)

Deliveries['Delivery_Date'] = pd.to_datetime(
    Deliveries['Delivery_Date']
)

Shipments['Shipment_Date'] = pd.to_datetime(
    Shipments['Shipment_Date']
)

ShipmentItems['Shipment_Date'] = pd.to_datetime(
    ShipmentItems['Shipment_Date']
)

In [8]:
OrderItems.head()

,Sales_Document,Item_Number,Material_Number,Quantity,Net_Price,Item_Status,Delivery_Date
0,1000001,10,MAT001,100,50,Open,2025-02-08
1,1000001,20,MAT002,200,30,Open,2025-02-08
2,1000002,10,MAT003,150,40,Delivered,2025-02-09
3,1000003,10,MAT001,300,50,Open,2025-02-10
4,1000004,10,MAT004,500,25,Closed,2025-02-12


In [10]:
Customers.to_csv("Customers.csv",index=False)

Carriers.to_csv("Carriers.csv",index=False)

Orders.to_csv("Orders.csv",index=False)

OrderItems.to_csv("OrderItems.csv",index=False)

Deliveries.to_csv("Deliveries.csv",index=False)

DeliveryItems.to_csv("DeliveryItems.csv",index=False)

Shipments.to_csv("Shipments.csv",index=False)

ShipmentItems.to_csv("ShipmentItems.csv",index=False)


In [11]:
# Create Customer Fulfillment Fact Table

FactFulfillment = Orders.merge(
    OrderItems,
    on="Sales_Document",
    how="left"
)

In [12]:
FactFulfillment = FactFulfillment.merge(
    Deliveries[
        [
            'Sales_Document',
            'Delivery_Number',
            'Delivery_Date',
            'Delivery_Status'
        ]
    ],
    on='Sales_Document',
    how='left'
)

In [16]:
FactFulfillment.head()

,Sales_Document,Order_Date,Customer_ID,Order_Type,Sales_Organization,Distribution_Channel,Division,Order_Status,Item_Number,Material_Number,...,Net_Price,Item_Status,Delivery_Date_x,Delivery_Number,Delivery_Date_y,Delivery_Status,Shipment_Number,Shipment_Date,Carrier,Shipment_Status
0,1000001,2025-02-01,CUST001,OR,1000,10,1,Open,10,MAT001,...,50,Open,2025-02-08,1001001.0,2025-02-10,Open,2001001.0,2025-02-10,Carrier1,In Transit
1,1000001,2025-02-01,CUST001,OR,1000,10,1,Open,20,MAT002,...,30,Open,2025-02-08,1001001.0,2025-02-10,Open,2001001.0,2025-02-10,Carrier1,In Transit
2,1000002,2025-02-02,CUST002,OR,1000,20,1,Delivered,10,MAT003,...,40,Delivered,2025-02-09,1001002.0,2025-02-11,Delivered,2001002.0,2025-02-11,Carrier2,Delivered
3,1000003,2025-02-05,CUST003,OR,1000,10,1,Open,10,MAT001,...,50,Open,2025-02-10,1001003.0,2025-02-12,Open,2001003.0,2025-02-12,Carrier1,In Transit
4,1000004,2025-02-06,CUST004,OR,1000,10,1,Closed,10,MAT004,...,25,Closed,2025-02-12,1001004.0,2025-02-13,Delivered,2001004.0,2025-02-13,Carrier3,Delivered


In [15]:
FactFulfillment = FactFulfillment.merge(
    Shipments[
        [
            'Sales_Document',
            'Shipment_Number',
            'Shipment_Date',
            'Carrier',
            'Shipment_Status'
        ]
    ],
    on='Sales_Document',
    how='left'
)

In [17]:
FactFulfillment = FactFulfillment.merge(
    Customers[
        [
            'Customer_ID',
            'Customer_Name',
            'Country',
            'Region'
        ]
    ],
    on='Customer_ID',
    how='left'
)

In [19]:
FactFulfillment.to_csv(
"FactFulfillment.csv",
index=False
)

# Don't Import it in PowerBi , as it is for experimental purpose only , i have created it for myself . remember it to create it in power bi.